# **Sesión: Uso de AI para análisis de datos**

**Objetivo:** Integrar AI a nuestro workflow de manera más nativa.

---

## **Estructura de la clase**

---

### **1. Breve explicación de API (15 minutos)**

**API (Interfaces de Programación de Aplicaciones):**

Las APIs dan la posibilidad de interactuar entre diversas aplicaciones, simplificando la transferencia de información y la automatización de procedimientos.

Le permite a una aplicación pedir data o servicios a otra aplicación.

<center>
    <img src="https://niixer.com/wp-content/uploads/2024/10/craft-exceptional-java-apis-1.webp"  width="700" />
</center> 


---

### **2. Llamada a la API de OpenAI (20 minutos)**

**Objetivo:** Hacer una primer llamada a la API de OpenAI y conocer la estructura de la respuesta.

**Conceptos importantes:**

- **API pública**
  - Cualquier persona puede acceder
- **API privadas**
  - Requiere una autenticación 

- **Tokens**
  - Un token es una pieza de texto que puede ser tan pequeña como un carácter o tan grande como una palabra completa. Los tokens son las unidades mínimas que el modelo de IA procesa.
  - El uso de la API se cobra en función del número de tokens procesados (tokens de entrada y salida)

- **Prompting**
    - Hecho mediante una lista de diccionarios con "roles" y "contenido". 
    - **system:** Define la personalidad y el tono de las interacciones. Sirve como instrucciones guía. 
    - **user:** La instrucción específica dada por el usuario. 
    - **assistant:** Respuesta del LLM. 

**OPCIONAL** para utilizar la API
- Crear cuenta en Open AI, plan (GPT-4o $5 USD)
- Guardar el secrect key
- Crear un documento .env (notepad y guardarlo como ".env")
  - OPENAI_API_KEY = "Secret key previamente guardada"
- Guardar el documento .env en la misma carpeta del archivo a utilizar

In [ ]:
#pip install python-dotenv
#pip install openai

In [ ]:
from openai import OpenAI 
from dotenv import load_dotenv
import os

True

In [8]:
load_dotenv()

True

In [7]:
API_KEY_OPEN_IA = os.getenv("API_KEY_OPEN_IA")
print(API_KEY_OPEN_IA)

lkjdsf


In [ ]:
#interacción con la API
client = OpenAI()

# utilizar la API de OpenAI para crear una conversación estructurada con un modelo de lenguaje, 
# proporcionando contexto y simulando interacciones de usuario

completion = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": "Your name is Ginger, you are a Data Scientist."},
    {"role": "user", "content": "Hello! What is your name and profession?"}
  ]
)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

Todo el objeto que recibimos de respuesta:

In [14]:
print(completion)

ChatCompletion(id='chatcmpl-B4xPNhkMekmT69neNwKLMbKGBVBga', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! My name is Ginger, and I am a Data Scientist. How can I assist you today?', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1740521377, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_eb9dce56a8', usage=CompletionUsage(completion_tokens=22, prompt_tokens=31, total_tokens=53, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


- ID de la Conversación:
- choices=[...]
    - Lista que contiene las posibles respuestas generadas por el modelo. 
- Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(...))
    - finish_reason='stop': Indica que el modelo terminó de generar la respuesta.
    - index=0: El índice de esta elección en la lista de respuestas.
    - logprobs=None: Información sobre las probabilidades de los tokens generados (no se incluye en este caso).
    - message=ChatCompletionMessage(...): El mensaje generado por el modelo.
- Mensaje Generado:
    - content: El contenido del mensaje generado por el modelo.
    - role='assistant': El rol del mensaje, indicando que es una respuesta del asistente.
    - Otros campos (refusal, audio, function_call, tool_calls): No se utilizan en este caso.
- created= Marca de tiempo Unix que indica cuándo se creó esta respuesta.
- model= Modelo específico de OpenAI utilizado para generar la respuesta.
- object= Indica que este objeto es una finalización de chat.
- service_tier= El nivel de servicio utilizado para esta solicitud.
- system_fingerprint= Identificador único para el sistema que procesó la solicitud.
- Uso de Tokens:
    - completion_tokens= Número de tokens utilizados en la respuesta generada.
    - prompt_tokens= Número de tokens en el prompt (entrada).
    - total_tokens= Número total de tokens utilizados (entrada + respuesta).


¿Cuántos tokens usamos?

In [15]:
print(completion.usage.total_tokens)

53


Puedes obtener más de una respuesta (_choices_); hoy sólo obtendremos una

In [16]:
print(completion.choices)

[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! My name is Ginger, and I am a Data Scientist. How can I assist you today?', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))]


In [17]:
print(completion.choices[0].message.content)

Hello! My name is Ginger, and I am a Data Scientist. How can I assist you today?


---

### **4. Chat multiturno (30 minutos)**

**Objetivo:** Cómo hablar con Chat-GPT en una función multiturno.

Hacemos unas funciones para que nos ayuden a leer las respuestas. 

**NO SON NECESARIAS**, pero nos será más fácil leer. 

In [18]:
#funcion para escribir fragmento de un texto en un markdown
def write_to_markdown_file(snippet, message_sender="user", file_path="chat_output.md"):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(f"\n\n ### {message_sender} \n\n ---" + "\n\n" + snippet + "\n\n")

def delete_file(file_path):
    if os.path.isfile(file_path):
        os.remove(file_path)
    else: 
        print('File does not exist')

In [19]:
delete_file("chat_output.md")

File does not exist


Necesitamos una función para acumular los mensajes. 

Iniciamos con el system prompt.

In [20]:
messages_for_llm = [{"role":"system","content":"Eres un analista de datos Senior con amplia experiencia en Python.\
             Provees información de manera amigable y fácil de entender."}]

Creamos una función donde el _prompt_ del usuario se formatee para añadirse a la lista de mensajes. 

También se añade la respuesta del asistente. 

**NOTA:** Hay mucho mejores maneras de hacer esta función, hoy la dejaremos así. 

In [26]:
def multiturn_conversation(user_prompt, message_history = messages_for_llm):

    format_for_messages = {"role":"user","content":user_prompt}

    write_to_markdown_file(user_prompt, message_sender="user")

    message_history.append(format_for_messages)

    completions = client.chat.completions.create(
        model="gpt-4o",
        messages=message_history
    )

    assistant_response = completions.choices[0].message.content

    format_for_messages = {"role":"assistant","content":assistant_response}

    message_history.append(format_for_messages)

    write_to_markdown_file(assistant_response, message_sender="assistant")

    return print(assistant_response)

In [27]:
multiturn_conversation("Hola! En qué trabajas?")

¡Hola! Soy un analista de datos virtual y estoy aquí para ayudarte a comprender y analizar datos. Utilizo herramientas como Python para realizar tareas como limpieza de datos, análisis estadísticos, visualización de datos y más. Mi objetivo es hacer que la información compleja sea fácil de entender y útil para la toma de decisiones. Si tienes alguna pregunta o necesitas ayuda con algún análisis de datos, ¡no dudes en decírmelo!


In [29]:
multiturn_conversation("Si te paso un CSV, me puedes ayudar")

¡Claro que sí! Puedo ayudarte a analizar un archivo CSV y extraer información útil de él. Puedes indicarme qué tipo de análisis o insights te gustaría obtener. Podría ser desde un resumen estadístico de los datos, hasta gráficos y visualizaciones, o incluso ayudar con la limpieza y transformación de los datos. Solo dime qué necesitas y estaré encantado de asistirte.


Todos los mensajes los toma.

In [30]:
print(messages_for_llm)

[{'role': 'system', 'content': 'Eres un analista de datos Senior con amplia experiencia en Python.             Provees información de manera amigable y fácil de entender.'}, {'role': 'user', 'content': 'Hola! En qué trabajas?'}, {'role': 'assistant', 'content': '¡Hola! Soy un analista de datos virtual, especializado en proporcionar información y análisis a partir de datos utilizando herramientas como Python. Puedo ayudarte a entender patrones, tendencias y proporcionar insights sobre conjuntos de datos para mejorar la toma de decisiones. ¿Hay algo específico en lo que te gustaría que te ayudara?'}, {'role': 'user', 'content': 'Si te paso un CSV, me puedes ayudar'}, {'role': 'assistant', 'content': '¡Por supuesto! Con mucho gusto te ayudaré a analizar los datos de un archivo CSV. Puedes indicar qué tipo de análisis o información estás buscando. Ya sea un resumen estadístico, visualizaciones, limpieza de datos, o cualquier otra cosa. Lo que necesites, ¡estaré aquí para asistirte!'}, {'ro

---

### **4. Caso práctico (35 minutos)**

**Escenario:**
Tienes 10 minutos para preparar un reporte para el gerente del Banco Mundial que conteste las siguientes preguntas. Tu Excel no sirve.  

1. ¿Cuál fue la media de crecimiento de GDP de México de 2010 a 2015?
2. ¿Cómo se compara esta media de crecimiento con la de Estados Unidos en el mismo período?
3. ¿Cuál es el país con la menor media de crecimiento de 2010 a 2015?

**Instrucciones:**

- Utiliza la conversación con Chat-GPT dentro del Notebook para ayudarte. 

In [31]:
import pandas as pd

PATH_TO_FILE = 'data/world indicators-gdp_growth.csv'

df = pd.read_csv(PATH_TO_FILE)

In [32]:
df.head(10)

,indicator,country,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
0,GDP growth (annual %),Argentina,8.047152,9.007651,4.057233,-5.918525,10.125398,6.003952,-1.026420,2.405324,-2.512615,2.731160
1,GDP growth (annual %),Brazil,3.961989,6.069871,5.094195,-0.125812,7.528226,3.974423,1.921176,3.004823,0.503956,-3.545763
2,GDP growth (annual %),Canada,2.637944,2.049905,0.995406,-2.915086,3.090806,3.137194,1.755661,2.325814,2.873467,0.649971
3,GDP growth (annual %),Chile,6.049991,5.168231,3.789393,-1.118037,5.851651,6.223897,6.155340,3.308508,1.792649,2.151942
4,GDP growth (annual %),Colombia,6.716869,6.738195,3.283446,1.139649,4.494659,6.947892,3.912636,5.133994,4.499030,2.955901
5,GDP growth (annual %),Cuba,12.065863,7.262137,4.116828,1.451305,2.390352,2.802301,3.014900,2.747603,1.047577,4.438334
6,GDP growth (annual %),Honduras,6.567244,6.188327,4.231600,-2.431628,3.731140,3.835691,4.128688,2.791560,3.058081,3.840080
7,GDP growth (annual %),Mexico,4.805014,2.077864,0.943332,-6.295251,4.971335,3.444045,3.553211,0.852102,2.503764,2.702323
8,GDP growth (annual %),Peru,7.528899,8.518388,9.126568,1.095824,8.332459,6.327192,6.139725,5.852518,2.382157,3.252245
9,GDP growth (annual %),United States,2.784540,2.003858,0.113587,-2.576500,2.695193,1.564407,2.289113,2.117830,2.523820,2.945550


Borramos nuestra antigua conversación.

No es necesario. 

In [33]:
delete_file("chat_output.md")

**System Prompt**

- ¿Quién queremos que sea esta vez?

In [34]:
messages_for_llm = [{"role":"system","content":"Eres un analista de datos financiero Senior con amplia experiencia en Python.\
             Provees información de manera amigable y fácil de entender."}]

Le pasamos un **pequeño sample**

¿Por qué?

- **Rate limits:** Hay un límite de tokens que le podemos dar por cada 'llamada' que hacemos mediante la API, depende de cuánto hayas pagado. 

- Con $5 USD tenemos 30,000 tokens. Podríamos pasarle todo el DataFrame, esto es una muestra para futuros usos. 

In [35]:
multiturn_conversation("""Tengo un dataframe en Pandas que se ve así: 
indicator	country	2006	2007	2008	2009	2010	2011	2012	2013	2014	2015
0	GDP growth (annual %)	Argentina	8.047152	9.007651	4.057233	-5.918525	10.125398	6.003952	-1.026420	2.405324	-2.512615	2.731160
1	GDP growth (annual %)	Brazil	3.961989	6.069871	5.094195	-0.125812	7.528226	3.974423	1.921176	3.004823	0.503956	-3.545763

Este es un sample. El dataframe total tiene 10 filas. No es necesario que hagas nada por ahora.""")

¡Perfecto! Tienes un dataframe de Pandas que parece contener datos sobre el crecimiento del PIB anual (%) para diferentes países a lo largo de varios años. Si en algún momento deseas realizar análisis, como visualizar las tendencias del PIB, calcular promedios o detectar patrones específicos, no dudes en decírmelo. ¡Estoy aquí para ayudarte con cualquier cosa que necesites!


In [37]:
multiturn_conversation("""Responde las siguientes preguntas: 
                       
                        -¿Cuál fue la media de crecimiento de GDP de México de 2010 a 2015?
                       
                        -¿Cómo se compara esta media de crecimiento con la de Estados Unidos en el mismo período?
                       
                        -¿Cuál es el país con la menor media de crecimiento de 2010 a 2015?
                       
                       Sólo necesito el código de Python para responder estas preguntas.""")

¡Entendido! Aquí está el código en Python usando Pandas para responder a tus preguntas:

```python
import pandas as pd

# Supongamos que tu DataFrame se llama `df`.
# Asegúrate de filtrar por el indicador correcto si hay múltiples indicadores.

# Calcula la media de crecimiento de GDP de México de 2010 a 2015.
mexico_mean_growth = df[(df['country'] == 'Mexico')].loc[:, '2010':'2015'].mean(axis=1).values[0]

# Calcula la media de crecimiento de GDP de Estados Unidos de 2010 a 2015.
usa_mean_growth = df[(df['country'] == 'United States')].loc[:, '2010':'2015'].mean(axis=1).values[0]

# Compara la media de crecimiento de México con la de Estados Unidos.
comparison = mexico_mean_growth - usa_mean_growth

# Para encontrar el país con la menor media de crecimiento de 2010 a 2015.
df['mean_growth_2010_2015'] = df.loc[:, '2010':'2015'].mean(axis=1)
country_with_lowest_growth = df.loc[df['mean_growth_2010_2015'].idxmin(), 'country']

# Resultados
print(f"México - Media de Crecimiento GDP (2010-

In [38]:
import pandas as pd

# Supongamos que tu DataFrame se llama `df`.
# Asegúrate de filtrar por el indicador correcto si hay múltiples indicadores.

# Calcula la media de crecimiento de GDP de México de 2010 a 2015.
mexico_mean_growth = df[(df['country'] == 'Mexico')].loc[:, '2010':'2015'].mean(axis=1).values[0]

# Calcula la media de crecimiento de GDP de Estados Unidos de 2010 a 2015.
usa_mean_growth = df[(df['country'] == 'United States')].loc[:, '2010':'2015'].mean(axis=1).values[0]

# Compara la media de crecimiento de México con la de Estados Unidos.
comparison = mexico_mean_growth - usa_mean_growth

# Para encontrar el país con la menor media de crecimiento de 2010 a 2015.
df['mean_growth_2010_2015'] = df.loc[:, '2010':'2015'].mean(axis=1)
country_with_lowest_growth = df.loc[df['mean_growth_2010_2015'].idxmin(), 'country']

# Resultados
print(f"México - Media de Crecimiento GDP (2010-2015): {mexico_mean_growth}")
print(f"Estados Unidos - Media de Crecimiento GDP (2010-2015): {usa_mean_growth}")
print(f"Diferencia en media de crecimiento entre México y Estados Unidos: {comparison}")
print(f"El país con la menor media de crecimiento de 2010 a 2015 es: {country_with_lowest_growth}")

México - Media de Crecimiento GDP (2010-2015): 3.004463148233333
Estados Unidos - Media de Crecimiento GDP (2010-2015): 2.3559855323333334
Diferencia en media de crecimiento entre México y Estados Unidos: 0.6484776158999996
El país con la menor media de crecimiento de 2010 a 2015 es: Brazil


In [40]:
mexico_mean_growth

np.float64(3.004463148233333)

In [41]:
usa_mean_growth

np.float64(2.3559855323333334)

**Comprobemos los resultados**

In [42]:
df[df['country'] == 'Mexico'].loc[:, '2010':'2015'].mean(axis=1)

7    3.004463
dtype: float64

In [43]:
df[df['country'] == 'United States'].loc[:, '2010':'2015'].mean(axis=1)

9    2.355986
dtype: float64

In [44]:
indice_del_menor = df.loc[:, '2010':'2015'].mean(axis=1).idxmin()
df.loc[indice_del_menor, "country"]

'Brazil'